# Proyek 01: Random Forest untuk Klasifikasi Anomali dan Serangan (UNSW-NB15)
**Mata Kuliah:** Machine Learning for Network Data, D4 Teknologi Rekayasa Komputer Jaringan, Universitas Bung Hatta

Notebook ini adalah **Tahap 1 (Modeling)** dari alur proyek:
**Dataset → Modeling → Save Model → App UI (Streamlit) → Deploy (GitHub + Streamlit Cloud) → Live App**

Yang dihasilkan notebook ini (folder `artifacts/`, diunduh sebagai `artifacts.zip`):
| File | Isi |
|---|---|
| `model/rf_binary.joblib` | Pipeline Random Forest: normal (0) vs serangan (1) |
| `model/rf_multiclass.joblib` | Pipeline Random Forest: kategori serangan (`attack_cat`) |
| `model/metadata.json` | Daftar fitur, nilai default, metrik, feature importance |
| `data/sample_flows.csv` | 300 flow contoh dari data test untuk mencoba aplikasi |
| `requirements.txt` | Versi library yang sama dengan Colab (penting saat deploy) |

> Jalankan: **Runtime > Run all**. Total sekitar 5 sampai 10 menit di CPU Colab.

## Identitas Mahasiswa
**Ujian Tengah Semester**, Mata Kuliah Machine Learning for Network Data. Isi data di sel bawah sebelum menjalankan *Run all*, supaya identitas ikut tercetak di PDF.

In [ ]:
# ===== ISI IDENTITAS ANDA DI SINI =====
IDENTITAS = {
    "Nama"          : "ISI NAMA LENGKAP",
    "NPM"           : "ISI NPM",
    "Program Studi" : "D4 Teknologi Rekayasa Komputer Jaringan",
    "Universitas"   : "Universitas Bung Hatta",
    "Mata Kuliah"   : "Machine Learning for Network Data",
    "Judul"         : "Implementasi Random Forest untuk Klasifikasi Anomali dan Serangan pada Dataset UNSW-NB15",
    "Link GitHub"   : "ISI SETELAH PUSH KE GITHUB",
    "Link Live App" : "ISI SETELAH DEPLOY (https://....streamlit.app)",
}
from datetime import datetime
print("=" * 70)
for k, v in IDENTITAS.items():
    print(f"{k:15s}: {v}")
print(f"{'Dijalankan':15s}: {datetime.now():%d %B %Y, %H:%M}")
print("=" * 70)

## 1. Import library dan cek versi

In [ ]:
import os, sys, glob, json, time, zipfile, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn, joblib
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
print("Python      :", sys.version.split()[0])
print("scikit-learn:", sklearn.__version__)
print("numpy       :", np.__version__)
print("pandas      :", pd.__version__)

## 2. Unduh dataset UNSW-NB15
Urutan pencarian file:
1. Folder lokal `data/` (bila Anda menjalankan di JupyterLab dan sudah punya file CSV).
2. Kaggle `mrwellsdavid/unsw-nb15` lewat `kagglehub` (sudah tersedia di Colab). Bila diminta login, jalankan `kagglehub.login()` dan isi username + API key (Kaggle > Settings > API > Create New Token).
3. Bila semuanya gagal: unggah manual `UNSW_NB15_training-set.csv` dan `UNSW_NB15_testing-set.csv` lewat dialog upload.

**Mengganti dataset?** Ubah bagian ini dan bagian 4 (nama kolom fitur dan target).

In [ ]:
TRAIN_FILE, TEST_FILE = "UNSW_NB15_training-set.csv", "UNSW_NB15_testing-set.csv"

def find_file(root, name):
    hits = glob.glob(os.path.join(root, "**", name), recursive=True)
    return hits[0] if hits else None

train_path = find_file("data", TRAIN_FILE)
test_path = find_file("data", TEST_FILE)

if not (train_path and test_path):
    try:
        import kagglehub
        path = kagglehub.dataset_download("mrwellsdavid/unsw-nb15")
        print("Folder Kaggle:", path)
        train_path, test_path = find_file(path, TRAIN_FILE), find_file(path, TEST_FILE)
    except Exception as e:
        print("Unduh Kaggle gagal:", e)

if not (train_path and test_path):
    print("Silakan unggah kedua file CSV secara manual.")
    from google.colab import files
    files.upload()
    train_path, test_path = TRAIN_FILE, TEST_FILE

print("Train:", train_path)
print("Test :", test_path)

## 3. Muat data dan eksplorasi singkat

In [ ]:
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
print("Train:", train_df.shape, "| Test:", test_df.shape)
train_df.head()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
train_df["label"].map({0: "normal", 1: "serangan"}).value_counts().plot.bar(ax=ax[0], color=["#C0392B", "#2E75B6"])
ax[0].set_title("Label biner (train)"); ax[0].tick_params(axis="x", rotation=0)
train_df["attack_cat"].value_counts().plot.barh(ax=ax[1], color="#172451")
ax[1].set_title("Kategori serangan (train)"); ax[1].invert_yaxis()
plt.tight_layout(); plt.show()

print("Nilai kosong per kolom (yang > 0):")
print(train_df.isna().sum()[lambda s: s > 0])

## 4. Fitur dan pipeline preprocessing
- Kolom `id` dibuang (hanya nomor urut).
- Fitur kategori `proto`, `service`, `state` di-one-hot. Kategori yang muncul < 100 kali digabung jadi "infrequent", dan kategori baru di data test tidak membuat error.
- Fitur numerik dipakai apa adanya: Random Forest **tidak butuh scaling** (berbeda dengan SVM dan K-Means).
- Seluruh preprocessing berada di dalam pipeline, jadi aplikasi Streamlit cukup memberi data mentah.

In [ ]:
CAT_COLS = ["proto", "service", "state"]
DROP_COLS = ["id", "attack_cat", "label"]
NUM_COLS = [c for c in train_df.columns if c not in CAT_COLS + DROP_COLS]
FEATURES = CAT_COLS + NUM_COLS

X_train, X_test = train_df[FEATURES], test_df[FEATURES]
y_train_bin, y_test_bin = train_df["label"], test_df["label"]
y_train_cat, y_test_cat = train_df["attack_cat"], test_df["attack_cat"]

def make_pipeline(**rf_overrides):
    """Pipeline = OneHotEncoder (kolom kategori) + RandomForest. Hyperparameter bisa diganti lewat argumen,
    mis. make_pipeline(max_depth=10). Tanpa argumen = konfigurasi awal dari template."""
    prep = ColumnTransformer(
        [("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=100), CAT_COLS)],
        remainder="passthrough")
    params = dict(n_estimators=100, max_depth=20, min_samples_leaf=2,
                  class_weight=None, n_jobs=-1, random_state=RANDOM_STATE)
    params.update(rf_overrides)
    return Pipeline([("prep", prep), ("rf", RandomForestClassifier(**params))])

print(f"{len(FEATURES)} fitur: {len(CAT_COLS)} kategori + {len(NUM_COLS)} numerik")

## 5. Model A: klasifikasi biner (normal vs serangan)
Hyperparameter awal: 100 pohon, `max_depth=20`, `min_samples_leaf=2`. Kedalaman dibatasi agar ukuran file model kecil (< 25 MB, batas upload lewat web GitHub).

In [ ]:
rf_bin = make_pipeline()
t0 = time.time(); rf_bin.fit(X_train, y_train_bin); train_time_bin = time.time() - t0

proba_bin = rf_bin.predict_proba(X_test)[:, 1]
pred_bin = (proba_bin >= 0.5).astype(int)
metrics_bin = {
    "accuracy": accuracy_score(y_test_bin, pred_bin),
    "precision": precision_score(y_test_bin, pred_bin),
    "recall": recall_score(y_test_bin, pred_bin),
    "f1": f1_score(y_test_bin, pred_bin),
    "roc_auc": roc_auc_score(y_test_bin, proba_bin),
    "train_time_s": train_time_bin,
}
print(json.dumps({k: round(v, 4) for k, v in metrics_bin.items()}, indent=2))
print(classification_report(y_test_bin, pred_bin, target_names=["normal", "serangan"], digits=3))

fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
ConfusionMatrixDisplay(confusion_matrix(y_test_bin, pred_bin), display_labels=["normal", "serangan"]).plot(
    ax=ax[0], cmap="Blues", colorbar=False, values_format="d")
ax[0].set_title("Confusion matrix (data test)")
RocCurveDisplay.from_predictions(y_test_bin, proba_bin, ax=ax[1], name="Random Forest")
ax[1].set_title("ROC curve"); plt.tight_layout(); plt.show()

## 6. Model B: klasifikasi multikelas (kategori serangan)
Target `attack_cat` berisi 10 kelas (Normal + 9 kategori serangan). Perhatikan **F1 per kelas**: kelas kecil seperti Worms dan Analysis jauh lebih sulit.

In [ ]:
rf_cat = make_pipeline()
t0 = time.time(); rf_cat.fit(X_train, y_train_cat); train_time_cat = time.time() - t0
pred_cat = rf_cat.predict(X_test)
metrics_cat = {
    "accuracy": accuracy_score(y_test_cat, pred_cat),
    "macro_f1": f1_score(y_test_cat, pred_cat, average="macro"),
    "weighted_f1": f1_score(y_test_cat, pred_cat, average="weighted"),
    "train_time_s": train_time_cat,
}
print(json.dumps({k: round(v, 4) for k, v in metrics_cat.items()}, indent=2))
print(classification_report(y_test_cat, pred_cat, digits=3))

classes = list(rf_cat.classes_)
cm = confusion_matrix(y_test_cat, pred_cat, labels=classes, normalize="true")
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", xticklabels=classes, yticklabels=classes)
plt.xlabel("Prediksi"); plt.ylabel("Aktual"); plt.title("Confusion matrix ternormalisasi (recall per kelas)")
plt.tight_layout(); plt.show()

## 7. Feature importance
Fitur mana yang paling dipakai Random Forest untuk memisahkan normal dan serangan?

In [ ]:
feat_names = rf_bin.named_steps["prep"].get_feature_names_out()
feat_names = [f.split("__", 1)[1] for f in feat_names]
imp = pd.Series(rf_bin.named_steps["rf"].feature_importances_, index=feat_names).sort_values(ascending=False)

imp.head(15).sort_values().plot.barh(figsize=(8, 6), color="#2E75B6")
plt.title("15 fitur terpenting (model biner)"); plt.xlabel("Mean decrease in impurity")
plt.tight_layout(); plt.show()
imp.head(10).round(4)

## 8. Eksperimen hyperparameter
Rubrik meminta **minimal 3 variasi hyperparameter** dibandingkan dalam satu tabel (metrik, waktu latih, ukuran model).
Eksperimen dilakukan pada **model biner** saja agar waktunya singkat (sekitar 3 sampai 6 menit untuk 4 variasi tambahan).
Baris *Baseline* memakai model `rf_bin` dari bagian 5, jadi tidak dilatih ulang.

Model yang disimpan di bagian 9 tetap **baseline** supaya ukuran file aman untuk GitHub dan konsisten dengan `app.py`.
Bila Anda ingin memakai variasi lain sebagai model final, latih ulang variasi itu ke `rf_bin` sebelum bagian 9.

In [ ]:
import io

def ukuran_mb(model):
    """Ukuran file model (MB) bila disimpan dengan joblib compress=3, sama seperti di bagian 9."""
    buf = io.BytesIO(); joblib.dump(model, buf, compress=3); return buf.tell() / 1e6

def evaluasi_biner(model, waktu_latih):
    p = model.predict_proba(X_test)[:, 1]
    yhat = (p >= 0.5).astype(int)
    return {"accuracy": accuracy_score(y_test_bin, yhat), "precision": precision_score(y_test_bin, yhat),
            "recall": recall_score(y_test_bin, yhat), "f1": f1_score(y_test_bin, yhat),
            "roc_auc": roc_auc_score(y_test_bin, p), "waktu_latih_s": waktu_latih,
            "ukuran_model_MB": ukuran_mb(model)}

variasi = {
    "B. Dangkal (depth=10)"                  : dict(max_depth=10),
    "C. Lebih banyak pohon (200 pohon)"      : dict(n_estimators=200),
    "D. class_weight='balanced'"             : dict(class_weight="balanced"),
    "E. Lebih dalam (depth=30, leaf=1)"      : dict(max_depth=30, min_samples_leaf=1),
}

hasil = {"A. Baseline (100 pohon, depth=20, leaf=2)": evaluasi_biner(rf_bin, train_time_bin)}
for nama, params in variasi.items():
    m = make_pipeline(**params)
    t0 = time.time(); m.fit(X_train, y_train_bin); dur = time.time() - t0
    hasil[nama] = evaluasi_biner(m, dur)
    print(f"selesai: {nama} ({dur:.0f} detik)")

tabel_eksperimen = pd.DataFrame(hasil).T.round(4)
tabel_eksperimen.index.name = "Variasi"
display(tabel_eksperimen)

ax = tabel_eksperimen[["accuracy", "recall", "f1", "roc_auc"]].plot.bar(figsize=(10, 4), rot=15, ylim=(0.7, 1.0))
ax.set_title("Perbandingan variasi hyperparameter (model biner, data test)")
plt.tight_layout(); plt.show()

## 9. Simpan model dan file pendukung aplikasi
Semua file disimpan di folder `artifacts/` dengan struktur yang sama dengan repo GitHub.
`requirements.txt` ditulis dengan versi **scikit-learn dan numpy yang sama dengan Colab**, karena model joblib
hanya bisa dibuka dengan versi yang cocok.

In [ ]:
os.makedirs("artifacts/model", exist_ok=True)
os.makedirs("artifacts/data", exist_ok=True)

joblib.dump(rf_bin, "artifacts/model/rf_binary.joblib", compress=3)
joblib.dump(rf_cat, "artifacts/model/rf_multiclass.joblib", compress=3)

# Fitur yang ditampilkan di form input manual aplikasi: 8 fitur numerik terpenting
top_numeric = [f for f in imp.index if f in NUM_COLS][:8]
metadata = {
    "dataset": "UNSW-NB15 (Kaggle mrwellsdavid/unsw-nb15)",
    "features": FEATURES, "cat_cols": CAT_COLS, "num_cols": NUM_COLS,
    "defaults": {**{c: float(X_train[c].median()) for c in NUM_COLS},
                 **{c: str(X_train[c].mode()[0]) for c in CAT_COLS}},
    "num_ranges": {c: [float(X_train[c].min()), float(X_train[c].quantile(0.99))] for c in NUM_COLS},
    "cat_options": {c: sorted(X_train[c].value_counts().head(15).index.astype(str).tolist()) for c in CAT_COLS},
    "top_numeric": top_numeric,
    "importance_top15": {k: float(v) for k, v in imp.head(15).items()},
    "classes_multiclass": classes,
    "metrics_binary": {k: float(v) for k, v in metrics_bin.items()},
    "metrics_multiclass": {k: float(v) for k, v in metrics_cat.items()},
    "versions": {"python": sys.version.split()[0], "scikit-learn": sklearn.__version__, "numpy": np.__version__},
}
with open("artifacts/model/metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

# 300 flow contoh (stratified per kategori) untuk dicoba di aplikasi
sample = test_df.sample(frac=1, random_state=RANDOM_STATE).groupby("attack_cat").head(30)
sample[FEATURES + ["attack_cat", "label"]].to_csv("artifacts/data/sample_flows.csv", index=False)

with open("artifacts/requirements.txt", "w") as f:
    f.write(f"streamlit\nscikit-learn=={sklearn.__version__}\nnumpy=={np.__version__}\npandas\njoblib\n")

for root, _, fs in os.walk("artifacts"):
    for fn in fs:
        p = os.path.join(root, fn); print(f"{p:45s} {os.path.getsize(p)/1e6:7.2f} MB")

In [ ]:
# Uji: muat ulang model dari file, lalu prediksi 5 flow contoh
m = joblib.load("artifacts/model/rf_binary.joblib")
s = pd.read_csv("artifacts/data/sample_flows.csv").head(5)
print(pd.DataFrame({"aktual": s["attack_cat"], "prob_serangan": m.predict_proba(s[FEATURES])[:, 1].round(3)}))

## 10. Unduh hasil
Jalankan sel di bawah untuk mengunduh `artifacts.zip`. Isinya disalin ke repo GitHub Anda
(folder `model/`, `data/`, dan file `requirements.txt`), bersama `app.py` dari template proyek.

In [ ]:
with zipfile.ZipFile("artifacts.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, fs in os.walk("artifacts"):
        for fn in fs:
            p = os.path.join(root, fn); z.write(p, os.path.relpath(p, "artifacts"))
try:
    from google.colab import files
    files.download("artifacts.zip")
except ImportError:
    print("Bukan di Colab: artifacts.zip tersimpan di folder kerja.")

## 11. Pertanyaan untuk laporan
1. Mengapa akurasi multikelas jauh lebih rendah daripada akurasi biner? Kelas mana yang paling sering tertukar?
2. Apa arti recall serangan bagi administrator jaringan? Berapa serangan yang lolos (false negative)?
3. Bandingkan 3 fitur terpenting dengan pengetahuan Anda tentang trafik jaringan. Masuk akal?
4. Apa pengaruh `max_depth` terhadap ukuran file model, waktu latih, dan metrik?

In [ ]:
# Angka untuk menjawab pertanyaan laporan, semuanya dihitung dari model di notebook ini
tn, fp, fn, tp = confusion_matrix(y_test_bin, pred_bin).ravel()
print("=== Model biner (threshold 0.5) ===")
print(f"True Negative  : {tn:6d}  (normal dikenali normal)")
print(f"False Positive : {fp:6d}  (normal dikira serangan = alarm palsu)")
print(f"False Negative : {fn:6d}  (serangan lolos)")
print(f"True Positive  : {tp:6d}  (serangan terdeteksi)")
print(f"Recall serangan: {tp/(tp+fn):.3%} | Recall normal: {tn/(tn+fp):.3%}")

print("\n=== Model multikelas ===")
cm_abs = confusion_matrix(y_test_cat, pred_cat, labels=classes)
pasangan = [(classes[i], classes[j], int(cm_abs[i, j])) for i in range(len(classes))
            for j in range(len(classes)) if i != j and cm_abs[i, j] > 0]
print("Lima pasangan yang paling sering tertukar (aktual -> prediksi):")
for a, p, n in sorted(pasangan, key=lambda x: -x[2])[:5]:
    print(f"  {a:15s} -> {p:15s} : {n} flow")

rep = pd.DataFrame(classification_report(y_test_cat, pred_cat, output_dict=True, zero_division=0)).T
rep = rep.loc[classes, ["precision", "recall", "f1-score", "support"]].sort_values("f1-score")
print("\nF1 per kelas, dari yang terendah:")
display(rep.round(3))

print("\n=== Tiga fitur terpenting (model biner) ===")
print(imp.head(3).round(4).to_string())

## 12. Dokumentasi penggunaan AI
Sesuai aturan proyek, bagian yang dibantu AI dicatat dengan jujur (isi dan sesuaikan dengan yang benar-benar Anda lakukan):

| Tool | Dipakai untuk | Contoh prompt |
|---|---|---|
| Claude (versi gratis) | Menambah sel identitas, tabel eksperimen hyperparameter, dan sel analisis jawaban pertanyaan laporan | *ISI prompt Anda* |

Seluruh angka di notebook ini dihasilkan dari run notebook sendiri, bukan dari jawaban AI.

**Sitasi dataset:** Moustafa, N., & Slay, J. (2015). *UNSW-NB15: a comprehensive data set for network intrusion detection systems.* MilCIS 2015.